# Sudoku Solver using SMT (Z3)

## 1. Sudoku Puzzle

9x9 grid of integers. `0` represents an empty cell that the solver needs to fill.

In [21]:
import random

def generate_sudoku(num_clues=30):
    # Pattern for a valid base Sudoku solution
    def pattern(r, c):
        return (r * 3 + r // 3 + c) % 9

    def shuffled(items):
        items = list(items)
        random.shuffle(items)
        return items

    # Randomize rows, columns, and numbers while preserving Sudoku validity
    rows = [
        g * 3 + r
        for g in shuffled(range(3))
        for r in shuffled(range(3))
    ]

    cols = [
        g * 3 + c
        for g in shuffled(range(3))
        for c in shuffled(range(3))
    ]

    nums = shuffled(range(1, 10))

    # Create a complete valid Sudoku board
    board = [
        [nums[pattern(r, c)] for c in cols]
        for r in rows
    ]

    # Remove cells until only num_clues remain
    cells_to_remove = 81 - num_clues

    positions = [(r, c) for r in range(9) for c in range(9)]
    random.shuffle(positions)

    for r, c in positions[:cells_to_remove]:
        board[r][c] = 0

    return board


def print_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-" * 21)

        row_str = ""

        for c in range(9):
            if c % 3 == 0 and c != 0:
                row_str += "| "

            row_str += f"{board[r][c] if board[r][c] != 0 else '.'} "

        print(row_str)


# Generate a new random puzzle each time
puzzle = generate_sudoku(num_clues=30)

print("Puzzle:")
print_board(puzzle)

Puzzle:
. 3 . | . 2 4 | 9 6 . 
. . . | 9 . 6 | . 8 . 
. . . | 1 . . | . 4 . 
---------------------
. 5 . | 4 . . | . 7 1 
. . 2 | . . 7 | . . 5 
. 1 7 | . 5 . | . . 9 
---------------------
. . . | 3 4 5 | . 9 . 
3 . . | 2 . . | . . 8 
. 6 9 | . . . | . . . 


## 2. Sudoku Solver

In [22]:
from z3 import Int, Solver, Distinct, And, sat

# describe what a a solved sudoku board looks like
def solve_sudoku_smt(puzzle):
    
    # creates a 9x9 matrix of z3 integer variables (one for each cell in the sudoku board)
    X = []
    for r in range(9):
        row = []
        for c in range(9):
            cell = Int(f"x_{r}_{c}")
            row.append(cell)
        X.append(row)

    # creates an empty solver (will add constraints to it later)
    s = Solver()

    # rule #1: every cell must be an integer between 1-9 (inclusive) 
    s.add([
        And(1 <= X[r][c], X[r][c] <= 9) 
        for r in range(9)
        for c in range(9)
    ])


    # rule #2: no duplicates in a row (all 9 cells are distinct)
    s.add([
        Distinct(X[r])
        for r in range(9)
    ])

    # rule 3: no duplicates in a column (similar to rule 2, but for columns)
    # only difference is we build the list by taking column c out of every row
    s.add([
        Distinct([X[r][c] for r in range(9)])
        for c in range(9)
    ])

    # rule #4: no duplicated in any 3x3 box (similar to rules 2 and 3, but for boxes) 
    # br = box row, bc = box column, r = row within box, c = column within box
    s.add([
        Distinct([
            X[3 * br + r][3 * bc + c] # cell in box (br, bc) at position (r, c) inside that box
            for r in range(3)
            for c in range(3)
        ])
        for br in range(3)
        for bc in range(3)
    ])

    # rule #5: given the rules above, z3 can produce any valid sudoku board. 
    # our goal is to solve a specific puzzle, so we have to tell z3 what the puzzle looks like 
    # if the cell in the puzzle is not zero, then we add a constraint of the corresponding value
    # if the cell in the puzzle is zero, we don't add any constraint and z3 can assign any value 
    s.add([
        X[r][c] == puzzle[r][c]
        for r in range(9)
        for c in range(9)
        if puzzle[r][c] != 0
    ])

    # s.check() asks z3 to find a solution to the constraints we added to the solver. 
    # if it returns sat then z3 found a solution. 
    if s.check() != sat:
        return None  

 
    # after s.check() found a solution, we can read the values of the variables in the model to get the solved soduo board
    model = s.model()
    return [[model.evaluate(X[r][c]).as_long() for c in range(9)] for r in range(9)]

## 3. Run it

In [23]:
solution = solve_sudoku_smt(puzzle)
print("Solved:")
print_board(solution)

Solved:
1 3 8 | 5 2 4 | 9 6 7 
5 7 4 | 9 3 6 | 1 8 2 
9 2 6 | 1 7 8 | 5 4 3 
---------------------
8 5 3 | 4 9 2 | 6 7 1 
4 9 2 | 6 1 7 | 8 3 5 
6 1 7 | 8 5 3 | 4 2 9 
---------------------
7 8 1 | 3 4 5 | 2 9 6 
3 4 5 | 2 6 9 | 7 1 8 
2 6 9 | 7 8 1 | 3 5 4 


## 4. Double-check the answer

A quick sanity check confirms every row, column, and 3x3 box really does contain the digits 1-9 exactly once.

In [20]:
def is_valid_sudoku(board):
    target = set(range(1, 10))

    for r in range(9):
        if set(board[r]) != target:
            return False
    for c in range(9):
        if {board[r][c] for r in range(9)} != target:
            return False
    for box_r in range(3):
        for box_c in range(3):
            cells = {board[box_r * 3 + i][box_c * 3 + j] for i in range(3) for j in range(3)}
            if cells != target:
                return False
    return True

print("Valid solution?", is_valid_sudoku(solution))

Valid solution? True
